# NYC 311 Maintenance Priority Modeling

This notebook walks through a maintenance-priority model using public NYC 311/HPD housing service requests. The public data is not a private work-order system, but it is close enough to practice the same analytics pattern: clean the raw requests, engineer features, define a review label, train a model, and inspect results.

## 1. Setup

The notebook imports the same data and modeling utilities used by the command-line scripts.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'scripts'))
sys.path.insert(0, str(ROOT / 'src'))

from download_data import build_training_table, download_public_requests
from maintenance_priority import LogisticPriorityModel, binary_metrics, build_feature_matrix

## 2. Download And Prepare Public Service Requests

The processed table contains public HPD complaint records from NYC 311. The target is a priority-follow-up proxy based on open status, closure time, and complaint category.

In [ ]:
processed_path = ROOT / 'data' / 'processed' / 'nyc_311_hpd_priority_training.csv'

if processed_path.exists():
    requests_df = pd.read_csv(processed_path)
else:
    raw_requests = download_public_requests()
    requests_df = build_training_table(raw_requests)
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    requests_df.to_csv(processed_path, index=False)

requests_df.head()

## 3. Summary Measures

I check complaint volumes, class balance, and closure-time behavior before training. This gives me a practical sense of the data quality.

In [ ]:
summary = {
    'rows': len(requests_df),
    'complaint_types': requests_df['complaint_type'].nunique(),
    'boroughs': requests_df['borough'].nunique(),
    'priority_rate': requests_df['priority_followup'].mean(),
    'median_closure_days': requests_df['closure_days'].median(),
}

pd.Series(summary).to_frame('value')

In [ ]:
requests_df['complaint_type'].value_counts().head(12).to_frame('request_count')

## 4. Data Wrangling And Feature Engineering

The feature matrix one-hot encodes complaint type, borough, and reporting channel. It also keeps simple operational fields such as month, winter flag, and descriptor availability.

In [ ]:
features = build_feature_matrix(requests_df)
target = requests_df['priority_followup'].astype(int)

features.head()

In [ ]:
feature_summary = features.agg(['mean', 'std']).T.sort_values('mean', ascending=False)
feature_summary.head(15).round(3)

## 5. Correlation Review

Correlation is a quick screening tool here. I use it to see which engineered fields move with the priority-follow-up proxy.

In [ ]:
corr_frame = features.copy()
corr_frame['priority_followup'] = target
target_corr = corr_frame.corr(numeric_only=True)['priority_followup'].sort_values(ascending=False)
target_corr.head(15).round(3)

In [ ]:
plot_corr = target_corr.drop('priority_followup').head(12).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(plot_corr.index, plot_corr.values)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature correlation with priority-follow-up proxy')
plt.xlabel('correlation')
plt.tight_layout()
plt.show()

## 6. Train/Test Split And Logistic Model

The model is a transparent logistic regression trained from scratch. I use the same deterministic split as the script so results are repeatable.

In [ ]:
test_mask = (requests_df.index % 4) == 0

X_train = features.loc[~test_mask].to_numpy()
X_test = features.loc[test_mask].to_numpy()
y_train = target.loc[~test_mask].to_numpy()
y_test = target.loc[test_mask].to_numpy()

model = LogisticPriorityModel()
model.fit(X_train, y_train)

preds = model.predict(X_test)
metrics = binary_metrics(y_test, preds)
metrics

## 7. Model Interpretation

Because the model is linear after scaling, the weights give a straightforward way to inspect what is pushing a request toward or away from priority review.

In [ ]:
weights = (
    pd.DataFrame({'feature': features.columns, 'weight': model.weights_})
    .assign(abs_weight=lambda frame: frame['weight'].abs())
    .sort_values('abs_weight', ascending=False)
)

weights[['feature', 'weight']].head(15)

In [ ]:
top_weights = weights.head(12).sort_values('weight')

plt.figure(figsize=(8, 5))
plt.barh(top_weights['feature'], top_weights['weight'])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top maintenance-priority model weights')
plt.xlabel('scaled coefficient')
plt.tight_layout()
plt.show()

## 8. Prediction Distribution

The probability distribution helps tune the threshold later. A threshold should be selected with operations input, not just by accepting the default `0.50`.

In [ ]:
probabilities = model.predict_proba(X_test)

plt.figure(figsize=(8, 4))
plt.hist(probabilities, bins=30, edgecolor='black')
plt.title('Predicted priority-follow-up probability')
plt.xlabel('probability')
plt.ylabel('request count')
plt.tight_layout()
plt.show()

## 9. Save A Model Payload

The command-line training script writes this JSON payload to `artifacts/maintenance_priority_model.json`.

In [ ]:
payload = model.to_dict(features.columns, metrics)
payload.keys()